In [0]:
# This is a library for working with a large amount of data at once- it can enact the same transformations on multiple dataframes with ease.
# Markdown is not used in this file because of the way the %run command works- the final goal is to have this library packaged and imported as a .py file instead of a notebook

In [0]:
%run ./Functions

In [0]:
from pyspark.sql.functions import from_unixtime, col, to_date, date_format, when, coalesce, array, lit, size, concat, expr, avg, stddev, mean
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType

class DataDictionary:
    def __init__(self, initial_dfs=None):
        if initial_dfs is None:
            self._dfs = {}
        else:
            self._dfs = dict(initial_dfs)
    
    def __getitem__(self, key):
        return self._dfs[key]
    
    def __setitem__(self, key, value):
        self._dfs[key] = value
    
    def __delitem__(self, key):
        del self._dfs[key]
    
    def __contains__(self, key):
        return key in self._dfs
    
    def __len__(self):
        return len(self._dfs)
    
    def __iter__(self):
        return iter(self._dfs)
    
    def __repr__(self):
        return f"{self.__class__.__name__}({self._dfs})"
    
    def pop(self, key):
        return self._dfs.pop(key)
    
    def keys(self):
        return self._dfs.keys()
    
    def values(self):
        return self._dfs.values()
    
    def items(self):
        return self._dfs.items()
    
    def get(self, key, default=None):
        return self._dfs.get(key, default)
    
    def update(self, other):
        self._dfs.update(other)
    
    def clear(self):
        self._dfs.clear()

    """
        Calculates what subset of dataframes are in the given subset and not in the exclude list.
        
        Parameters:
            subset (list): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list): a list of strings representing the keys of dataframes to exclude from the subset.
    """
    def subsetCalc(self, subset, exclude):
        if not subset:
            subset = self.keys()
        return list(set(subset) - set(exclude))

    """
        Displays each dataframe with its key and the column names

        Parameters:
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def display(self, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        dfStrs = []
        for key in subset:
            dfStrs.append(f'{key}: {self[key].columns}')
        print("\n".join(dfStrs))

    """
        Loads all files in the given path into the data dictionary. Uses list_files() and getFileNames() from the Functions notebook to achieve this.

        Parameters:
            path (string): A string representing the filepath to load from
            search_dirs (bool, optional): Whether or not you want to search further directories found within the given directory. Defaults to True
            filetype (string, optional): The file extension to look for. Defaults to .csv

        Returns:
            self
    """
    def loadAllData(self, path, search_dirs=True, filetype=".csv", subset=[], exclude=[]):
        # Grab all files under the given path
        files = list_files(path, search_dirs, filetype)
        # Grab a list of the file names from the path
        fileNames = [f.replace(f"dbfs:{path}", "").replace(f"{filetype}/", "") for f in files]

        # Load all the files into the dataframe dictionary
        for i, p in enumerate(files):
            if (not subset or fileNames[i] in subset) and fileNames[i] not in exclude:
                if filetype == '.csv':
                    df = spark.read.options(header=True, inferSchema=True).csv(p)
                elif filetype == '.parquet':
                    print(f"Loading file: {fileNames[i]}")
                    df = spark.read.options(header=True, inferSchema=True).parquet(p)
                self[fileNames[i]] = df

        return self
    
    """
        Writes all dataframes into the given path. Uses the key as the name of the file. By default writes all dataframes.

        Parameters:
            path (string): A string representing the filepath to write to
            filetype (string, optional): The file extension to write for. Defaults to .csv
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.

        Returns:
            self
    """
    def writeAllData(self, path, filetype=".csv", subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        # Write all files
        for k in subset:
            if filetype == '.csv':
                self[k].write.options(header=True).mode('overwrite').csv(f'{path}{k}.csv')
            elif filetype == '.parquet':
                self[k].write.options(header=True).mode('overwrite').parquet(f'{path}{k}.parquet')

        return self
    
    """
        Gets all distinct column names between all columns in every dataframe. Useful for looking at what needs to be cleaned

        Parameters:
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.

        Returns:
            list of strings
    """
    def distinctCols(self, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        cols = []
        for key in subset:
            cols = cols + self[key].columns

        # Convert to a set then back to a list
        return list(set(cols))
    
    """
        Drops all the given columns in every dataframe, if it exists in that dataframe

        Parameters:
            *cols (str): The columns to drop
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.

        Returns:
            self
    """
    def dropAll(self, *cols, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        # For each column to be dropped, loop over the dictionary
        for col in cols:
            for key in subset:
                # If the dataframe has the column, drop it
                if(has_column(self[key], col)):
                    self[key] = self[key].drop(col)
        return self
    
    """
        Renames all given column names into the given new name.

        Parameters:
            newname (str): the new name for the columns
            *old_cols (str): the names of the columns to be renamed
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.

        Returns:
            self
    """
    def renameAll(self, newname, *old_cols, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        for key in subset:
            for col in old_cols:
                if(has_column(self[key], col)):
                    self[key] = self[key].withColumnRenamed(col, newname)
        return self
    
    """
        Similar to renameAll(), but instead takes a single old column and a list of new names. The list must be the same size as the number of dataframes. Due to the way this works, you must provide the same number of new names as the number of columns that have the old name. This is used if you want to rename multiple columns that initially share the same name into separate names. For example: You have 3 columns called Units. Rename all columns from Units into Units (length), Units (time), and Units (weight) in that order. The order is determined by the order shown in the .display() method.

        Parameters:
            newnames (list): list of strings for the new names
            old_col (str): the column to be renamed
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.

        Returns:
            self

        Throws:
            ValueError when the list of new names is not the same size as the number of dataframes
    """
    def renameSeparately(self, newnames, old_col, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)
            
        if len(newnames) != len(self.values()):
            raise ValueError(f"Expected {len(self.values())} new name values, got {len(newnames)}.")

        i = 0
        for k in self.keys():
            self[k] = self[k].withColumnRenamed(old_col, newnames[i])
            i += 1

        return self
    
    """
        Joins all dataframes together, and keeps the new. Assumes all dataframes have columns in the join key

        Parameters:
            name (str): key for the newly joined dataframe
            on (list): list of strings for the join key
            how (str, optional): the type of join to use. Defaults to outer.
            remove (bool, optional): whether to remove the original dataframe from the dictionary or not. Defaults to False.
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.

        Returns:
            self
    """
    def joinAll(self, name, on, how='outer', remove=False, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        # Get the first dataframe to be used in the join
        keys = list(subset)
        joined = self[keys[0]]
        if remove:
            del self[keys[0]]

        # Iterate over all other dataframes, joining them to the joined dataframe
        for key in keys[1:]:
            joined = joined.join(self[key], on=on, how=how)
            if remove:
                del self[key]
        
        # Add the joined dataframe to the dictionary and return self
        self[name] = joined
        return self
    
    """
        Unifies all given dataframes, stacking them on top of each other.

        Parameters:
            name (str): key for the newly joined dataframe
            remove (bool, optional): whether to remove the original dataframe from the dictionary or not. Defaults to False
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def unifyAll(self, name, remove=False, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        # Get the first dataframe to be unified
        unified = self[subset[0]]
        if remove:
            del self[subset[0]]
        
        # Iterate over all other dataframes, unioning them to the unified dataframe
        for key in subset[1:]:
            unified = unified.union(self[key])
            if remove:
                del self[key]
        
        # Add the unified dataframe to the dictionary and return self
        self[name] = unified
        return self
    
    """
        Displays all dataframes in the dictionary using pyspark's display()

        Parameters:
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def displayAll(self, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)
            
        for key in subset:
            print(f"Displaying: {key}")
            display(self[key])

    """
        Casts all columns in the given column to the given type across all dataframes.

        Parameters:
            col (str): the column to cast
            toType (str): the type to cast to
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def castAll(self, colName, toType, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        for key in subset:
            if has_column(self[key], colName):
                # Converts to date data type
                self[key] = self[key].withColumn(colName, col(colName).cast(toType))

    """
        Converts all unix timestamps in the given column into the given format across all dataframes.

        Parameters:
            colName (str): the column to convert
            format (str): the format to convert to
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def unixToDateAll(self, colName, format, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        for key in subset:
            if has_column(self[key], colName):
                # Converts to date data type
                self[key] = self[key].withColumn(colName, (col(colName)/1000).cast("timestamp").cast("date"))

    """
        Converts all timestamps in the given column into the given format across all dataframes.

        Parameters:
            colName (str): the column to convert
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def toDateAll(self, colName, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        for key in subset:
            self[key] = self[key].withColumn(colName, to_date(col(colName)))

    """
        Selects all columns in the given column across all dataframes.

        Parameters:
            colName (str): the column to select
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def selectAll(self, *colName, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        for key in subset:
            if has_column(self[key], colName):
                if isinstance(colName, list):
                    self[key] = self[key].select(*colName)
                else:
                    self[key] = self[key].select(colName)

    """
        Filters all dataframes based on the given column and value.

        Parameters:
            colName (str): the column to filter on
            value (str): the value to filter on
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the

        Returns:
            DataDictionary: a dictionary of filtered dataframes
    """
    def filterAll(self, colName, value, subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        filtered_dfs = DataDictionary()
        for key in subset:
            if has_column(self[key], colName):
                filtered_dfs[key] = self[key].filter(col(colName) == value)
        return filtered_dfs


    """
        Finds outliers in the given columns and dataframes using the interquartile range (IQR) method.

        Parameters:
            threshold (float): the threshold for outlier detection. Defaults to 1.5.
            partition_col (str, optional): the column to partition the dataframes by. Defaults to empty string.
            cols (list, optional): a list of strings representing the columns to check for outliers. Defaults to all columns.
            exclude_cols (list, optional): a list of strings representing the columns to exclude from the check. Defaults to empty list.
            subset (list, optional): a list of strings representing the keys of a subset of dataframes to use. Defaults to all keys.
            exclude (list, optional): a list of strings representing the keys of dataframes to exclude from the subset. Defaults to empty list.
    """
    def find_outliers(self, threshold=1.5, partition_col="", cols=[], exclude_cols=[], subset=[], exclude=[]):
        subset = self.subsetCalc(subset, exclude)

        outlierDfs = DataDictionary()
        for key in subset:
            # If the list of cols to check are empty, default to checking all columns
            dfCols = cols
            if not dfCols:
                dfCols = self[key].columns

            outlier_logs = []
            # Loop through the columns to check in the chosen dataframe
            for column in dfCols:
                if has_column(self[key], column) and isinstance(self[key].schema[column].dataType, (DoubleType, IntegerType)) and column not in exclude_cols:

                    # Calculate Q1 and Q3
                    if not partition_col:
                        mn = self[key].agg(mean(column)).first()[0]
                        stdv = self[key].agg(stddev(column)).first()[0]
                    else:
                        partition = Window.partitionBy(partition_col)
                        mn = avg(col(column)).over(partition)
                        stdv = stddev(col(column)).over(partition)
                    
                    # Determine outlier bounds
                    lower_bound = (mn - threshold * stdv)
                    upper_bound = (mn + threshold * stdv)
                    
                    # Create outlier log for the column
                    if not partition_col:
                        outlier_log = when(
                            (col(column) < lower_bound) | (col(column) > upper_bound),
                            concat(
                                lit(f"Outlier detected in column {column}. Value "),
                                col(column).cast("string"),
                                lit(f" violated bounds "),
                                lit(f" {lower_bound} to {upper_bound}"),
                            )
                        )
                    else:
                        outlier_log = when(
                            (col(column) < lower_bound) | (col(column) > upper_bound),
                            concat(
                                lit(f"Outlier detected in column {column} for partition "),
                                col(partition_col).cast("string"),
                                lit(f". Value "),
                                col(column).cast("string"),
                                lit(f" violated bounds "),
                                lower_bound.cast("string"),
                                lit(" to "),
                                upper_bound.cast("string"),
                            )
                        )
                    outlier_logs.append(outlier_log)

            # Create a new column with the outlier logs
            outlierDfs[key] = self[key].withColumn(
                'outliers',
                array(*[coalesce(log, lit(None)) for log in outlier_logs])
            )
            # Only include rows with outliers in them
            outlierDfs[key] = outlierDfs[key].withColumn(
                'outliers',
                expr("filter(outliers, x -> x IS NOT NULL)")
            ).filter(size(col('outliers')) > 0)
        
        return outlierDfs
